# Vision-Bot: Control gestual por Bluetooth

## Dr. Jesús Emmanuel Solís Pérez
### jsolisp@unam.mx

---

## DESCRIPCIÓN GENERAL: 
Visión-Bot es un taller intensivo y práctico diseñado para fusionar el poder de la Inteligencia Artificial con la robótica móvil. En este curso introductorio, los participantes construirán un sistema de control gestual desde cero. Aprenderán a utilizar la cámara de una computadora para capturar los movimientos de la mano en tiempo real, procesar los datos con Python y enviar comandos inalámbricos vía Bluetooth para controlar los motores de un carrito robótico.

# CONTENIDO DEL CURSO
## TEMARIO

1. Visión por computadora.
    * Uso de Python y MediaPipe para detectar los puntos de referencia de la mano.
2. Lógica de programación.
    * Clasificación de gestos simples.
3. Microcontroladores.
    * Configuración de un ESP32 para recibir comandos via Bluetooth.
4. Integración.
    * Envío de datos vía Bluetooth desde Python al carrito.
    * Pruebas de campo.

## HABILIDADES QUE APRENDERÁ

1. Uso y configuración de la librería MediaPipe para la detección de marcadores de la mano.
2. Normalización e interpretación de coordenadas espaciales en un flujo de video en vivo.
3. Estructuración de condicionales para la clasificación de gestos.
4. Transmisión de datos por Bluetooth.
5. Programación y configuración básica de microcontroladores.

## Requisitos de Software
1. **Python 3.8+**
2. **OpenCV** (para manipulación de video)

---

In [2]:
# Descomentar la siguiente línea si no tienes instalado pyserial.
# !pip install pyserial

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import serial  # Importar la librería pySerial
import time

In [3]:
# ==========================================
# 0. CONFIGURACIÓN DEL PUERTO BLUETOOTH (SPP)
# ==========================================
# Reemplaza 'COM3' por el puerto serie asignado a tu dispositivo Bluetooth en tu computadora.
# En Linux/macOS suele ser algo como '/dev/rfcomm0' o '/dev/tty.HC05-DevB'
PORT = 'COM3' 
BAUD_RATE = 115200  # Velocidad de transmisión (debe coincidir con la configuración del dispositivo Bluetooth)

In [4]:
try:
    bluetooth_conn = serial.Serial(PORT, BAUD_RATE, timeout=1)
    time.sleep(2)  # Espera a que se establezca la conexión física
    print(f"Conectado exitosamente al puerto Bluetooth: {PORT}")
except Exception as e:
    print(f"No se pudo conectar al puerto Bluetooth {PORT}: {e}")
    print("El programa continuará ejecutándose solo de forma visual.")
    bluetooth_conn = None

No se pudo conectar al puerto Bluetooth COM3: [Errno 2] could not open port COM3: [Errno 2] No such file or directory: 'COM3'
El programa continuará ejecutándose solo de forma visual.


In [5]:
# ==========================================
# 1. CONFIGURACIÓN DEL DETECTOR (MediaPipe Tasks)
# ==========================================

# Configurar las opciones base indicando la ruta del modelo descargado (HandLandmarker)
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')

# Nota de referencia para descargar el modelo si no se tiene localmente:
# wget -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task

# Definir las opciones específicas para el detector de manos utilizando la nueva API de visión
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE, # Modo de procesamiento por imagen individual
    num_hands=1,                           # Limitar la detección a un máximo de 1 mano
    min_hand_detection_confidence=0.7,     # Confianza mínima requerida para detectar una mano
    min_hand_presence_confidence=0.7       # Confianza mínima requerida para mantener el seguimiento
)

# Crear la instancia del detector a partir de las opciones configuradas
detector = vision.HandLandmarker.create_from_options(options)

# ==========================================
# 2. INICIALIZACIÓN DE LA CÁMARA
# ==========================================

# Inicializar la captura de video desde la cámara web predeterminada (índice 0)
cap = cv2.VideoCapture(0)
print("Iniciando detector de gestos moderno. Presiona 'q' para salir.")

# ==========================================
# 3. BUCLE PRINCIPAL DE PROCESAMIENTO
# ==========================================

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("No se pudo acceder a la cámara.")
        break

    # Voltear la imagen horizontalmente (efecto espejo) para una interacción más natural
    frame = cv2.flip(frame, 1)
    
    # MediaPipe requiere imágenes en formato RGB, OpenCV lee por defecto en BGR
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Crear un objeto de imagen compatible con la nueva estructura de MediaPipe
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    # Procesar el fotograma actual para detectar los puntos clave de la mano (landmarks)
    detection_result = detector.detect(mp_image)
    gesture = "Desconocido"
    command_to_send = None  # Carácter por defecto a enviar

    # Verificar si se han detectado manos en el fotograma
    if detection_result.hand_landmarks:
        for hand_landmarks in detection_result.hand_landmarks:
            # Extraer puntos clave de referencia (la nueva API devuelve coordenadas normalizadas entre 0.0 y 1.0)
            wrist = hand_landmarks[0]         # Muñeca
            index_tip = hand_landmarks[8]     # Punta del dedo índice
            middle_tip = hand_landmarks[12]   # Punta del dedo medio
            ring_tip = hand_landmarks[16]     # Punta del dedo anular
            pinky_tip = hand_landmarks[20]    # Punta del dedo meñique
            
            index_mcp = hand_landmarks[5]     # Nudillo base del índice
            middle_mcp = hand_landmarks[9]    # Nudillo base del medio
            ring_mcp = hand_landmarks[13]     # Nudillo base del anular
            pinky_mcp = hand_landmarks[17]    # Nudillo base del meñique
            
            # ----------------------------------
            # A. Detección del gesto "STOP"
            # ----------------------------------
            # Se comprueba si las puntas de los 4 dedos están más arriba (menor valor en Y) 
            # que sus respectivos nudillos base (MCP), indicando que la palma está abierta y frontal.
            fingers_extended = (
                index_tip.y < index_mcp.y and
                middle_tip.y < middle_mcp.y and
                ring_tip.y < ring_mcp.y and
                pinky_tip.y < pinky_mcp.y
            )
            
            if fingers_extended:
                gesture = "Stop"
            else:
                # ----------------------------------
                # B. Detección de Inclinaciones / Direcciones
                # ----------------------------------
                # Calcular la diferencia en el eje horizontal (X) y de profundidad (Z) 
                # entre el nudillo medio y la muñeca para determinar la inclinación de la mano.
                dx = middle_mcp.x - wrist.x
                dz = middle_mcp.z - wrist.z  # Coordenada de profundidad proporcionada por MediaPipe
                
                # Definir los umbrales de sensibilidad para los movimientos
                threshold_lat = 0.12
                threshold_depth = 0.05
                
                # Evaluar las condiciones espaciales
                if dx > threshold_lat:
                    gesture = "Derecha"
                    command_to_send = 'R'
                elif dx < -threshold_lat:
                    gesture = "Izquierda"
                    command_to_send = 'L'
                elif dz < -threshold_depth:
                    gesture = "Adelante"
                    command_to_send = 'F'
                elif dz > threshold_depth:
                    gesture = "Atrás"
                    command_to_send = 'B'
                else:
                    gesture = "Neutro"
                    command_to_send = 'S' # Detener si está en neutro

            # ----------------------------------
            # ENVÍO DE DATOS POR BLUETOOTH (SPP)
            # ----------------------------------
            if bluetooth_conn and bluetooth_conn.is_open and command_to_send:
                # Opcional: Enviar solo si el comando es diferente al anterior para no saturar
                if command_to_send != last_sent_gesture:
                    bluetooth_conn.write(command_to_send.encode('utf-8'))
                    print(f"Comando enviado por Bluetooth: {command_to_send}")
                    last_sent_gesture = command_to_send

            cv2.putText(frame, f"Gesto: {gesture}", (30, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
    else:
        # Si no se detecta mano, opcionalmente enviar parada
        if bluetooth_conn and bluetooth_conn.is_open and last_sent_gesture != 'S':
            bluetooth_conn.write(b'S')
            last_sent_gesture = 'S'

    # Mostrar el resultado visual en una ventana de OpenCV
    cv2.imshow('Detector de Gestos con MediaPipe', frame)

    # Interrupción del ciclo: si el usuario presiona la tecla 'q', se termina el bucle
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ==========================================
# 4. LIBERACIÓN DE RECURSOS
# ==========================================
if bluetooth_conn and bluetooth_conn.is_open:
    bluetooth_conn.close()
    print("Conexión Bluetooth cerrada.")
    
cap.release()
cv2.destroyAllWindows()

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1787008647.348635  179422 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787008647.366136  179423 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Iniciando detector de gestos moderno. Presiona 'q' para salir.


W0000 00:00:1787008648.150413  179423 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/opt/miniconda3/envs/research/lib/python3.13/site-packages/cv2/qt/plugins"
QFontDatabase: Cannot find font directory /opt/miniconda3/envs/research/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /opt/miniconda3/envs/research/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /opt/miniconda3/envs/research/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https